<a href="https://colab.research.google.com/github/TheZaraKhan/ai-reliability/blob/main/notebooks/bike_share_reliability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How reliable are AI data-analysis assistants?

This notebook evaluates whether language models can answer questions about
Toronto Bike Share trips by writing pandas code.


## Stage 1 — Data source and quality checks

Source: City of Toronto Open Data, Bike Share Toronto Ridership Data.

Licence: Open Government Licence – Toronto.

Scope: Trips that started from January 1 through March 31, 2024.

### 1.1 · Set up the official Toronto Bike Share data

This notebook uses trip records published by the City of Toronto. We load the 2024 archive and select trips that started from January 1 through March 31, 2024.

[City of Toronto Bike Share Ridership Data](https://open.toronto.ca/dataset/bike-share-toronto-ridership-data/)


In [42]:
import io
import json
import re
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import pandas as pd
from IPython.display import display

# Start with a tiny dataset whose expected results we can check by hand.
# Change this to False when you're ready to use the official Toronto data.
USE_FIXTURE = False

# Include January 1 and exclude April 1.
QUARTER_START = pd.Timestamp("2024-01-01")
QUARTER_END = pd.Timestamp("2024-04-01")

# CKAN is the API Toronto uses to list files in its open-data catalogue.
CKAN_URL = (
    "https://ckan0.cf.opendata.inter.prod-toronto.ca/api/3/action/"
    "package_show?id=bike-share-toronto-ridership-data"
)
ARCHIVE_NAME = "bikeshare-ridership-2024.zip"

# An empty value after the final comma represents a missing station name.
# Trip 11 is duplicated. Trip 12 starts in March and ends in April.
FIXTURE = """Trip_Id,Trip_Duration,Start_Time,End_Time,End_Station_Name
10,60,2023-12-31 23:59:30,2024-01-01 00:00:30,Station A
11,59,2024-01-01 00:00:00,2024-01-01 00:00:59,
11,59,2024-01-01 00:00:00,2024-01-01 00:00:59,
12,100000,2024-03-31 23:59:59,2024-04-02 03:46:39,Station B
13,60,2024-04-01 00:00:00,2024-04-01 00:01:00,Station B
"""

### 1.2 · Find and download the official dataset

The City of Toronto's CKAN API lists the files available for this dataset. The download function uses that list to find the 2024 ZIP, saves it under a temporary name, and checks that it contains a CSV.

Toronto publishes one ZIP for the full year. The next step selects the quarter we need.

[City of Toronto Bike Share Ridership Data](https://open.toronto.ca/dataset/bike-share-toronto-ridership-data/)

In [43]:
from shutil import copyfileobj


def download_archive(destination: Path) -> None:
    """Find the official 2024 file through CKAN and save it locally."""

    # Ask the catalogue which files belong to the Bike Share dataset.
    with urlopen(CKAN_URL, timeout=60) as response:
        package = json.load(response)

    if not package["success"]:
        raise RuntimeError("The City of Toronto CKAN request failed.")

    # Find the download URL for the 2024 ZIP.
    resources = package["result"]["resources"]
    resource = next(
        (item for item in resources if item["name"] == ARCHIVE_NAME),
        None,
    )
    if resource is None:
        raise RuntimeError(f"CKAN did not list {ARCHIVE_NAME}.")

    # Use a temporary filename so an interrupted download can't be
    # mistaken for the finished archive.
    partial = destination.with_suffix(".part")
    try:
        with urlopen(resource["url"], timeout=120) as response:
            with partial.open("wb") as output:
                copyfileobj(response, output)

        # Make sure the downloaded ZIP contains a CSV before renaming it.
        with ZipFile(partial) as archive:
            if not any(name.lower().endswith(".csv") for name in archive.namelist()):
                raise ValueError("The archive contains no CSV.")

        partial.replace(destination)
    finally:
        # Remove the temporary file if the download or archive check failed.
        partial.unlink(missing_ok=True)

### 1.3 · Inspect the columns and select the quarter

The loader displays the original column names and converts them to `snake_case`. For example, `Start_Time` becomes `start_time`. It then checks for the required fields and converts timestamps from text to dates.

A trip belongs to the first quarter if it started on or after January 1 and before April 1, 2024. We keep duplicate rows and unusual durations so we can measure how cleaning choices affect answers later.

In [44]:
def snake_case(name: str) -> str:
    """Convert a header like 'Start Time' or 'Start_Time' to 'start_time'."""
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def load_quarter(source) -> pd.DataFrame:
    """Read a CSV and return trips that started in the selected quarter."""

    # Read just the header first and show exactly how each name changes.
    original = pd.read_csv(source, nrows=0).columns.tolist()
    normalized = [snake_case(name) for name in original]
    display(pd.DataFrame({
        "original_header": original,
        "normalized_header": normalized,
    }))

    # Stop with a useful message if the file has a different schema.
    required = {"trip_id", "trip_duration", "start_time", "end_time"}
    missing = required - set(normalized)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    if len(normalized) != len(set(normalized)):
        raise ValueError("Two source columns normalize to the same name.")

    # Rewind after reading the header, then process the large CSV in batches.
    source.seek(0)
    quarter_chunks = []

    for chunk in pd.read_csv(source, chunksize=200_000):
        chunk.columns = normalized

        # Convert start times from text to datetimes so we can compare dates.
        # An unexpected date format raises an error you can inspect.
        starts = pd.to_datetime(
            chunk["start_time"],
            format="%Y-%m-%d %H:%M:%S",
            errors="raise",
        )

        # The start time decides which quarter a trip belongs to.
        in_quarter = starts.ge(QUARTER_START) & starts.lt(QUARTER_END)
        if not in_quarter.any():
            continue

        # Keep every row in the quarter, including short and duplicate trips.
        selected = chunk.loc[in_quarter].copy()
        selected["start_time"] = starts.loc[in_quarter]
        selected["end_time"] = pd.to_datetime(
            selected["end_time"],
            format="%Y-%m-%d %H:%M:%S",
            errors="raise",
        )
        quarter_chunks.append(selected)

    if not quarter_chunks:
        raise ValueError("No trips found for the selected quarter.")

    # Join the selected batches into one DataFrame with a fresh row index.
    return pd.concat(quarter_chunks, ignore_index=True)

### 1.4 · Preview the loaded data

The source label tells us whether we loaded fictional practice data or official Toronto records. The first few rows show what a trip record looks like, and the DataFrame summary shows the column types and how many values each column contains.

In practice mode, we expect three rows. Trip 11 appears twice, and trip 12 is included because it started in March.

In [45]:


if USE_FIXTURE:
    print("Using synthetic practice data.")
    # StringIO lets pandas read the practice text like a CSV file.
    df = load_quarter(io.StringIO(FIXTURE))
else:
    print("Using City of Toronto data for 2024 Q1.")
    # Reuse the ZIP if it's already in the current Colab session.
    archive_path = Path(ARCHIVE_NAME)
    if not archive_path.exists():
        download_archive(archive_path)

    with ZipFile(archive_path) as archive:
        # Inspect the archive instead of guessing the CSV filename.
        csv_names = [
            name for name in archive.namelist() if name.lower().endswith(".csv")
        ]
        print("CSV files in archive:", csv_names)

        if len(csv_names) != 1:
            raise ValueError("Expected one CSV. Check the filenames printed above.")

        with archive.open(csv_names[0]) as source:
            df = load_quarter(source)

# head() previews the first five rows. info() shows data types and non-null counts.
display(df.head())
df.info()

Using City of Toronto data for 2024 Q1.
CSV files in archive: ['bikeshare-ridership-2024.csv']


,original_header,normalized_header
0,Trip_Id,trip_id
1,Trip_Duration,trip_duration
2,Start_Station_Id,start_station_id
3,Start_Time,start_time
4,Start_Station_Name,start_station_name
5,End_Station_Id,end_station_id
6,End_Time,end_time
7,End_Station_Name,end_station_name
8,Bike_Id,bike_id
9,User_Type,user_type


,trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,bike_model
0,26682740,528,7041,2024-01-01 00:00:08,Edward St / Yonge St,7130.0,2024-01-01 00:08:56,Davenport Rd / Bedford Rd,7855,Casual,EFIT G5
1,26682742,1924,7245,2024-01-01 00:02:04,Tecumseth St / Queen St W - SMART,7259.0,2024-01-01 00:34:08,Lower Spadina Ave / Lake Shore Blvd W,2453,Casual,ICONIC
2,26682743,2741,7052,2024-01-01 00:03:25,Wellington St W / Bay St,7366.0,2024-01-01 00:49:06,Fort York Blvd / Bathurst St - SMART,7332,Member,ICONIC
3,26682744,353,7041,2024-01-01 00:03:39,Edward St / Yonge St,7418.0,2024-01-01 00:09:32,College Park - Yonge St Entrance,782,Member,ICONIC
4,26682746,2118,7721,2024-01-01 00:04:44,Widmer St / King St W,7721.0,2024-01-01 00:40:02,Widmer St / King St W,1008,Member,ICONIC


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 779266 entries, 0 to 779265
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   trip_id             779266 non-null  int64         
 1   trip_duration       779266 non-null  int64         
 2   start_station_id    779266 non-null  int64         
 3   start_time          779266 non-null  datetime64[ns]
 4   start_station_name  779266 non-null  object        
 5   end_station_id      778983 non-null  float64       
 6   end_time            779251 non-null  datetime64[ns]
 7   end_station_name    778983 non-null  object        
 8   bike_id             779266 non-null  int64         
 9   user_type           779266 non-null  object        
 10  bike_model          779266 non-null  object        
dtypes: datetime64[ns](2), float64(1), int64(4), object(4)
memory usage: 65.4+ MB


### 1.5 · Check data quality

The report counts rows, missing values, trips under 60 seconds, and duplicates. It also shows the shortest and longest recorded durations.

An identical row and a repeated trip ID are two separate checks. For both, we count occurrences after the first. In practice mode, the assertions check the report against values we can verify by hand.

In [46]:
# Read the recorded duration as a number of seconds.
duration = pd.to_numeric(df["trip_duration"], errors="raise")

# These values are calculated from whichever DataFrame you loaded.
quality = pd.DataFrame(
    [
        ("Rows", len(df)),
        ("Minimum duration (seconds)", duration.min()),
        ("Maximum duration (seconds)", duration.max()),
        ("Trips under 60 seconds", int(duration.lt(60).sum())),
        ("Duplicate rows after the first", int(df.duplicated().sum())),
        (
            "Repeated non-missing trip IDs after the first",
            int(df["trip_id"].dropna().duplicated().sum()),
        ),
    ],
    columns=["check", "value"],
)

# isna() marks missing cells. Summing those marks gives a count per column.
missing_values = df.isna().sum().rename("null_count").to_frame()

display(quality)
display(missing_values)

# These expected values come from the five practice rows above.
# Run these checks only in fixture mode.
if USE_FIXTURE:
    assert df["trip_id"].tolist() == [11, 11, 12]
    assert pd.api.types.is_datetime64_any_dtype(df["start_time"])
    assert pd.api.types.is_datetime64_any_dtype(df["end_time"])
    assert df["end_time"].max() == pd.Timestamp("2024-04-02 03:46:39")
    assert duration.min() == 59
    assert duration.max() == 100000
    assert int(duration.lt(60).sum()) == 2
    assert int(df.duplicated().sum()) == 1
    assert int(df["trip_id"].dropna().duplicated().sum()) == 1
    assert missing_values.loc["end_station_name", "null_count"] == 2
    print("Stage 1 fixture checks passed.")

,check,value
0,Rows,779266
1,Minimum duration (seconds),0
2,Maximum duration (seconds),2320126
3,Trips under 60 seconds,1572
4,Duplicate rows after the first,0
5,Repeated non-missing trip IDs after the first,0


,null_count
trip_id,0
trip_duration,0
start_station_id,0
start_time,0
start_station_name,0
end_station_id,283
end_time,15
end_station_name,283
bike_id,0
user_type,0


### 1.6 · What this dataset can answer

The official dataset supports questions about trip counts, recorded durations, station endpoints, bicycle models, and membership categories. It has no rider ID, age, gender, trip distance, or route field. A bicycle ID identifies a bicycle, so it cannot tell us how many distinct people rode.

The practice data checks that the loading and reporting code works. Results about Toronto ridership come from running the notebook on the official data.

## Stage 2 — Load the questions and answer key

Load the fixed set of 40 Toronto Bike Share questions and their reference
answers from `benchmark/questions.csv` on GitHub. The preview shows each
question’s ID, category, and text.

The [answer key notebook](https://github.com/TheZaraKhan/ai-reliability/blob/main/benchmark/build_answer_key.ipynb)
computes the reference answers from the official trip data using deterministic
pandas functions and exports them to the CSV.

Each model receives the question text, dataset schema, and sample rows.
The evaluation uses the reference answers to grade the results after execution.

In [47]:
import pandas as pd

# URL to point to the raw file instead of the GitHub HTML page viewer
csv_url = "https://raw.githubusercontent.com/TheZaraKhan/ai-reliability/main/benchmark/questions.csv"

# Keep blank answers as empty strings for unanswerable questions.
benchmark = pd.read_csv(csv_url, dtype=str, keep_default_na=False)

print(f"Loaded {len(benchmark)} questions from GitHub.")
display(benchmark[["question_id", "category", "question"]].head())

Loaded 40 questions from GitHub.


,question_id,category,question
0,Q01,simple,How many trip records are in the quarter?
1,Q02,simple,What is the shortest recorded trip duration in...
2,Q03,simple,What is the longest recorded trip duration in ...
3,Q04,simple,How many trips have a recorded duration under ...
4,Q05,simple,How many trips have a missing end station ID?


## Stage 3 — Collect and run model responses



### 3.1 · Configure the evaluation

Each model receives the same question, complete dataset schema, and five
sample rows. The full trip table is available during execution. Reference
answers and oracle code stay in the benchmark and are used later for grading.

The default run uses a deterministic mock model and makes no API calls.

In [48]:
import ast
import hashlib
import json
import math
import subprocess
import sys
import tempfile
import time
from pathlib import Path

MAX_USD = 1.00
MAX_OUTPUT_TOKENS = 1500
TIMEOUT_SECONDS = 30
OUTPUT_BYTES = 4000

# Live API calls require an explicit switch and verified prices.
ENABLE_LIVE = False
CACHE_DIR = Path("stage3_cache")

# Prices are USD per million tokens. Verify model IDs and prices before live use.
# None keeps a model disabled until you enter both prices.
MODELS = {
    "mock": {"id": "mock-v1"},
    "openai": {"id": "gpt-6-luna", "input_price": None, "output_price": None},
    "anthropic": {"id": "claude-haiku-5-5", "input_price": None, "output_price": None},
    "gemini": {"id": "gemini-3.8-flash", "input_price": None, "output_price": None},
}

# The grader expects these exact labels for ambiguous questions.
# This lookup contains output labels only. Reference values stay in the answer key.
ALLOWED_LABELS = {
    "Q17": ("start_time", "end_time"),
    "Q18": ("mean", "median"),
    "Q37": ("departures", "arrivals", "both"),
    "Q38": ("calendar_date", "day_of_week"),
}

# Use the question ID to look up its text without including the answer key.
question_by_id = benchmark.set_index("question_id")["question"]
schema_text = df.dtypes.astype(str).to_string()
sample_text = df.head(5).to_csv(index=False)


def build_prompt(question: str, question_id: str) -> str:
    """Build the same prompt for every provider."""
    labels = ALLOWED_LABELS.get(question_id, ())
    label_instruction = ""

    # Add an interpretation instruction only for questions that have labels.
    if labels:
        label_instruction = (
            f"For this question, print a dictionary with exactly one of these keys: "
            f"{', '.join(labels)}. Use the key that matches your interpretation "
            "and put your calculated result under that key.\n"
        )

    return f"""Answer using only the supplied Toronto Bike Share trip data.
It contains trips that started from January through March 2024.
Each row represents a trip record. bike_id identifies a bicycle.
user_type is a membership category.

The full pandas DataFrame will be available as df when your code runs.
Use these sample rows to understand the format.

Columns and data types:
{schema_text}

Sample rows:
{sample_text}

Question:
{question}

Return exactly one JSON object with four fields:
{{"can_answer": true, "reason_if_not": "", "code": "", "answer": ""}}

For an answerable question, set can_answer to true and provide short Python code
that uses df and prints one result. Use pandas if an import is needed.
For a question asking for raw and cleaned results, print a dictionary with
those two labels.
{label_instruction}Leave answer empty if you need the execution output to know the result.

If the available fields cannot answer the question, set can_answer to false,
explain the missing information in reason_if_not, and leave code and answer empty.
"""

### 3.2 · Collect and cache responses

Save each API response and its token usage locally before parsing it. Reuse
an existing response when the provider, model, prompt, and settings match.
The cost estimate uses configured token prices. Live requests stop when the
estimated budget is reached.

The mock returns predictable responses so we can check the pipeline without
API keys or paid requests.

In [49]:
def mock_reply(question_id: str) -> str:
    """Return known examples for checking the evaluation pipeline."""
    examples = {
        "Q01": (True, "", "print(len(df))", ""),
        # Deliberately wrong. Q04 asks for durations under 60 seconds.
        "Q04": (True, "", "print(len(df))", ""),
        "Q40": (False, "The trip records lack a rider identifier.", "", ""),
    }

    can_answer, reason, code, answer = examples[question_id]
    return json.dumps({
        "can_answer": can_answer,
        "reason_if_not": reason,
        "code": code,
        "answer": answer,
    })


def request_live(provider: str, model: str, prompt: str) -> tuple[str, int, int]:
    """Return raw response text, input tokens, and billable output tokens."""
    from google.colab import userdata

    if provider == "openai":
        from openai import OpenAI

        client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"), max_retries=0)
        reply = client.responses.create(
            model=model,
            input=prompt,
            temperature=0,
            reasoning={"effort": "none"},
            max_output_tokens=MAX_OUTPUT_TOKENS,
            store=False,
        )
        return reply.output_text, reply.usage.input_tokens, reply.usage.output_tokens

    if provider == "anthropic":
        from anthropic import Anthropic

        client = Anthropic(
            api_key=userdata.get("ANTHROPIC_API_KEY"),
            max_retries=0,
        )
        reply = client.messages.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            thinking={"type": "disabled"},
            max_tokens=MAX_OUTPUT_TOKENS,
        )
        text = "".join(block.text for block in reply.content if block.type == "text")
        return text, reply.usage.input_tokens, reply.usage.output_tokens

    if provider == "gemini":
        from google import genai
        from google.genai import types

        client = genai.Client(
            api_key=userdata.get("GEMINI_API_KEY"),
            http_options=types.HttpOptions(
                retry_options=types.HttpRetryOptions(attempts=1)
            ),
        )
        reply = client.models.generate_content(
            model=model,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0,
                max_output_tokens=MAX_OUTPUT_TOKENS,
                response_mime_type="application/json",
                thinking_config=types.ThinkingConfig(thinking_level="MINIMAL"),
            ),
        )
        usage = reply.usage_metadata
        output_tokens = (
            (usage.candidates_token_count or 0)
            + (usage.thoughts_token_count or 0)
        )
        return reply.text or "", usage.prompt_token_count, output_tokens

    raise ValueError(f"Unknown provider: {provider}")


def collect_response(provider: str, question_id: str) -> dict:
    """Reuse a cached response or make one new request within the budget."""
    config = MODELS[provider]

    # Pass the ID so build_prompt can look up the allowed interpretation labels.
    prompt = build_prompt(question_by_id.loc[question_id], question_id)

    cache_key = hashlib.sha256(json.dumps(
        [provider, config["id"], question_id, prompt, MAX_OUTPUT_TOKENS, "settings-v1"],
        ensure_ascii=False,
    ).encode("utf-8")).hexdigest()

    CACHE_DIR.mkdir(exist_ok=True)
    cache_path = CACHE_DIR / f"{cache_key}.json"
    if cache_path.exists():
        return {**json.loads(cache_path.read_text()), "cache_hit": True}

    if provider != "mock":
        if not ENABLE_LIVE:
            raise RuntimeError("Live API calls are disabled.")

        prices = (config["input_price"], config["output_price"])
        if any(price is None or not math.isfinite(price) or price < 0 for price in prices):
            raise ValueError("Enter verified input and output prices before making live calls.")

        spent = sum(
            json.loads(path.read_text())["estimated_usd"]
            for path in CACHE_DIR.glob("*.json")
        )

        # Estimate the next call's cost before sending it.
        # Provider token accounting and billing can differ from this estimate.
        reserved = (
            (len(prompt.encode("utf-8")) + 1024) * prices[0]
            + (2 * MAX_OUTPUT_TOKENS) * prices[1]
        ) / 1_000_000
        if spent >= MAX_USD or spent + reserved > MAX_USD:
            raise RuntimeError("Estimated API budget reached.")

    started = time.perf_counter()

    if provider == "mock":
        raw_text, input_tokens, output_tokens = mock_reply(question_id), 0, 0
        estimated_usd = 0.0
    else:
        raw_text, input_tokens, output_tokens = request_live(
            provider, config["id"], prompt
        )
        estimated_usd = (
            input_tokens * prices[0] + output_tokens * prices[1]
        ) / 1_000_000

    record = {
        "provider": provider,
        "model": config["id"],
        "question_id": question_id,
        "raw_response": raw_text,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "estimated_usd": estimated_usd,
        "latency_seconds": time.perf_counter() - started,
    }

    # Save the raw response even if parsing it later fails.
    temp_path = cache_path.with_suffix(".tmp")
    temp_path.write_text(json.dumps(record, indent=2), encoding="utf-8")
    temp_path.replace(cache_path)

    return {**record, "cache_hit": False}

### 3.3 · Run reviewed pandas code in a separate process

Validate the response format before execution. Run the script with a
30-second timeout, a pandas-only import check, a fresh temporary directory,
and an environment that does not inherit API keys. Limit captured output.

These checks do not create a true security sandbox. Review live model code
before running it. Mock scripts are written by us and can run directly.

In [50]:
def parse_reply(raw_text: str) -> dict:
    """Check that the response matches the experiment's JSON format."""
    reply = json.loads(raw_text)
    fields = {"can_answer", "reason_if_not", "code", "answer"}
    if not isinstance(reply, dict) or set(reply) != fields:
        raise ValueError("Expected exactly the four response fields.")
    if type(reply["can_answer"]) is not bool:
        raise ValueError("can_answer must be a JSON boolean.")
    if any(not isinstance(reply[field], str) for field in fields - {"can_answer"}):
        raise ValueError("The remaining fields must be strings.")
    if reply["can_answer"] and not reply["code"].strip():
        raise ValueError("An answerable response needs code.")
    if not reply["can_answer"] and (
        not reply["reason_if_not"].strip() or reply["code"].strip() or reply["answer"].strip()
    ):
        raise ValueError("An abstention needs a reason and empty code and answer fields.")
    return reply


def check_imports(code: str) -> None:
    """Reject explicit imports other than pandas."""
    for node in ast.walk(ast.parse(code)):
        if isinstance(node, ast.Import):
            if any(alias.name != "pandas" for alias in node.names):
                raise ValueError("Only pandas imports are allowed.")
        elif isinstance(node, ast.ImportFrom):
            if node.module != "pandas" or node.level:
                raise ValueError("Only pandas imports are allowed.")


def run_reviewed_code(code: str) -> dict:
    """Run code you have reviewed and capture its bounded output."""
    check_imports(code)

    with tempfile.TemporaryDirectory() as temp_dir:
        work = Path(temp_dir)
        # This file is produced from the trusted in-memory DataFrame.
        # Pickle preserves pandas types, including datetimes and missing values.
        df.to_pickle(work / "trips.pkl")

        # Limit the files used to capture stdout and stderr in Linux/Colab.
        prelude = (
            "import resource\n"
            "resource.setrlimit(resource.RLIMIT_FSIZE, (65536, 65536))\n"
            "import pandas as pd\n"
            "df = pd.read_pickle('trips.pkl')\n"
        )
        (work / "analysis.py").write_text(prelude + "\n" + code, encoding="utf-8")

        with (work / "stdout.txt").open("wb") as stdout:
            with (work / "stderr.txt").open("wb") as stderr:
                try:
                    process = subprocess.run(
                        [sys.executable, "-I", "analysis.py"],
                        cwd=work,
                        env={"HOME": temp_dir, "LANG": "C.UTF-8"},
                        stdin=subprocess.DEVNULL,
                        stdout=stdout,
                        stderr=stderr,
                        timeout=TIMEOUT_SECONDS,
                    )
                    status = "ok" if process.returncode == 0 else "execution_error"
                except subprocess.TimeoutExpired:
                    status = "timeout"

        def read_output(filename: str) -> str:
            with (work / filename).open("rb") as stream:
                data = stream.read(OUTPUT_BYTES + 1)
            text = data[:OUTPUT_BYTES].decode("utf-8", errors="replace")
            return text + ("\n[truncated]" if len(data) > OUTPUT_BYTES else "")

        return {
            "execution_status": status,
            "stdout": read_output("stdout.txt"),
            "stderr": read_output("stderr.txt"),
        }

### 3.4 · Verify the workflow with three mock cases

Q01 runs an ordinary trip count. Q04 deliberately prints the total row
count for a question about short trips. Q40 explains why distinct riders
cannot be counted from the available fields. The results below are
workflow checks. Model accuracy will be measured in Stage 4.

In [51]:
stage3_records = []

for question_id in ("Q01", "Q04", "Q40"):
    record = collect_response("mock", question_id)
    reply = parse_reply(record["raw_response"])

    if reply["can_answer"]:
        execution = run_reviewed_code(reply["code"])
    else:
        execution = {"execution_status": "abstained", "stdout": "", "stderr": ""}

    stage3_records.append({**record, **reply, **execution})

stage3_results = pd.DataFrame(stage3_records)
display(stage3_results[[
    "question_id", "model", "execution_status", "stdout", "reason_if_not",
    "input_tokens", "output_tokens", "estimated_usd", "cache_hit",
]])

# Check execution and caching without using the answer key.
checks = stage3_results.set_index("question_id")
assert checks.loc["Q01", "execution_status"] == "ok"
assert checks.loc["Q01", "stdout"].strip() == str(len(df))
assert checks.loc["Q04", "execution_status"] == "ok"
assert checks.loc["Q40", "execution_status"] == "abstained"
assert checks.loc["Q40", "stdout"] == ""
assert collect_response("mock", "Q01")["cache_hit"] is True

print("Stage 3 mock checks passed.")

,question_id,model,execution_status,stdout,reason_if_not,input_tokens,output_tokens,estimated_usd,cache_hit
0,Q01,mock-v1,ok,779266\n,,0,0,0.0,False
1,Q04,mock-v1,ok,779266\n,,0,0,0.0,False
2,Q40,mock-v1,abstained,,The trip records lack a rider identifier.,0,0,0.0,False


Stage 3 mock checks passed.


## Stage 4 — Grade model outputs

### 4.1 · Compare outputs with the answer key

Compare the result printed by the model’s pandas code with the reference
answer for the same question. Counts and IDs must match exactly. Calculated
values such as means and percentages use a 0.5% relative tolerance.

Cleaning questions require both `raw` and `cleaned` values. Ambiguous
questions require an accepted interpretation and its matching value.
Unanswerable questions require an abstention with a reason.

If the code prints the correct result and the model supplies a different
written answer, record the error as `misreported`.

In [52]:
import ast
import json
import math
import re

answer_key = benchmark.set_index("question_id")
RELATIVE_TOLERANCE = 0.005  # 0.5%

# These questions return counts or IDs even if the CSV represents them as
# decimals, for example "779266.0". Q38 returns a date or weekday label.
EXACT_NUMERIC_IDS = {
    "Q01", "Q02", "Q03", "Q04", "Q05", "Q06", "Q07", "Q11", "Q13",
    "Q15", "Q17",
    "Q21", "Q22", "Q23", "Q24", "Q25", "Q26", "Q27",
    "Q30", "Q31", "Q33", "Q35", "Q36", "Q37",
}

NUMBER = re.compile(
    r"[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d+)?(?:[eE][+-]?\d+)?"
    r"|[+-]?\.\d+(?:[eE][+-]?\d+)?"
)


def parse_value(text: str):
    """Read a number, date label, or dictionary from the CSV or model output."""
    text = text.strip()
    if len(text) > 4096:
        raise ValueError("Result is too long to grade.")

    # The CSV can contain Python-style dictionaries with single quotes.
    # literal_eval reads these as data without running them as code.
    if text.startswith("{"):
        try:
            value = json.loads(text)
        except json.JSONDecodeError:
            value = ast.literal_eval(text)
        if not isinstance(value, dict):
            raise ValueError("Expected a dictionary.")
        return value

    # A trailing percent sign leaves a value on the 0–100 scale.
    numeric_text = text.removesuffix("%").strip()
    if NUMBER.fullmatch(numeric_text):
        return float(numeric_text.replace(",", ""))

    # Preserve dates such as 2024-03 and labels such as Monday.
    return text


def scalar_matches(actual, expected, *, exact_numeric: bool) -> bool:
    """Compare one value using the rule for its question."""
    if isinstance(expected, str):
        expected = parse_value(expected)
    if isinstance(actual, str):
        actual = parse_value(actual)

    if type(expected) in (int, float):
        if type(actual) not in (int, float):
            return False
        if not math.isfinite(actual) or not math.isfinite(expected):
            return False
        if exact_numeric:
            return actual == expected
        return math.isclose(
            actual, expected, rel_tol=RELATIVE_TOLERANCE, abs_tol=0.0
        )

    return (
        isinstance(actual, str)
        and isinstance(expected, str)
        and actual.strip().casefold() == expected.strip().casefold()
    )


def matching_result(text: str, expected, category: str, question_id: str) -> bool:
    """Check a printed value or structured written answer."""
    try:
        actual = parse_value(text)
        exact_numeric = question_id in EXACT_NUMERIC_IDS

        if category == "messy_data":
            # Both fields are required and each must match its own reference.
            return (
                isinstance(actual, dict)
                and isinstance(expected, dict)
                and actual.keys() == expected.keys() == {"raw", "cleaned"}
                and all(
                    scalar_matches(actual[key], expected[key], exact_numeric=exact_numeric)
                    for key in ("raw", "cleaned")
                )
            )

        if category == "ambiguous":
            # The key identifies the interpretation. Check the answer for that key.
            if not isinstance(actual, dict) or not isinstance(expected, dict):
                return False
            if len(actual) != 1:
                return False
            key, value = next(iter(actual.items()))
            return key in expected and scalar_matches(
                value, expected[key], exact_numeric=exact_numeric
            )

        return scalar_matches(actual, expected, exact_numeric=exact_numeric)

    except (ValueError, TypeError, SyntaxError, OverflowError):
        return False


def grade_response(result) -> dict:
    """Grade one Stage 3 response and assign its primary error tag."""
    reference = answer_key.loc[result.question_id]
    category = reference["category"]
    code_correct = None
    written_answer_correct = None

    if category == "unanswerable":
        correct = (
            not result.can_answer
            and result.execution_status == "abstained"
            and bool(result.reason_if_not.strip())
        )
        tag = "" if correct else "hallucinated_answer"

    elif result.execution_status == "abstained":
        correct, tag = False, "wrong_result"

    elif result.execution_status != "ok":
        correct, tag = False, "execution_error"

    else:
        expected = parse_value(reference["computed_truth"])
        code_correct = matching_result(
            result.stdout, expected, category, result.question_id
        )

        if result.answer.strip():
            written_answer_correct = matching_result(
                result.answer, expected, category, result.question_id
            )

            # For an ambiguous question, the written answer and stdout
            # must also name the same interpretation.
            if category == "ambiguous" and code_correct and written_answer_correct:
                written_answer_correct = (
                    parse_value(result.answer).keys()
                    == parse_value(result.stdout).keys()
                )

        correct = code_correct and written_answer_correct is not False

        if not code_correct:
            tag = "wrong_result"
        elif written_answer_correct is False:
            tag = "misreported"
        else:
            tag = ""

    return {
        "question_id": result.question_id,
        "model": result.model,
        "category": category,
        "correct": correct,
        "error_tag": tag,
        "code_correct": code_correct,
        "written_answer_correct": written_answer_correct,
    }


# Grade the responses actually collected in Stage 3.
graded_results = pd.DataFrame(
    grade_response(result) for result in stage3_results.itertuples(index=False)
)
display(graded_results)

,question_id,model,category,correct,error_tag,code_correct,written_answer_correct
0,Q01,mock-v1,simple,True,,True,None
1,Q04,mock-v1,simple,False,wrong_result,False,None
2,Q40,mock-v1,unanswerable,True,,None,None


### 4.2 · Verify the grading rules

The three mock responses test a correct calculation, a wrong calculation,
and a correct abstention. Additional controlled cases check decimal tolerance,
required cleaning labels, ambiguous answers, and the `misreported`,
`execution_error`, and `hallucinated_answer` tags.

In [53]:
from types import SimpleNamespace

# Confirm that the three real mock outputs still get their expected grades.
mock_checks = graded_results.set_index("question_id")
assert bool(mock_checks.loc["Q01", "correct"])
assert mock_checks.loc["Q04", "error_tag"] == "wrong_result"
assert bool(mock_checks.loc["Q40", "correct"])

# Decimal tolerance accepts a rounded value and rejects a larger difference.
assert matching_result("10.57%", 10.565779, "multi_step", "Q10")
assert not matching_result("11%", 10.565779, "multi_step", "Q10")

# Both raw and cleaned values are required.
assert matching_result(
    '{"raw": 947.09, "cleaned": 818.44}',
    {"raw": 947.086575, "cleaned": 818.441428},
    "messy_data",
    "Q14",
)
assert not matching_result(
    '{"raw": 947.09}',
    {"raw": 947.086575, "cleaned": 818.441428},
    "messy_data",
    "Q14",
)

# Both interpretations are accepted, but each must have its exact count.
march = {"start_time": 312613, "end_time": 312585}
assert matching_result('{"start_time": 312613}', march, "ambiguous", "Q17")
assert matching_result('{"end_time": 312585}', march, "ambiguous", "Q17")
assert not matching_result('{"end_time": 312613}', march, "ambiguous", "Q17")

# Simulate correct code output paired with an incorrect written answer.
test_response = SimpleNamespace(
    question_id="Q01",
    model="grader-test",
    can_answer=True,
    execution_status="ok",
    stdout="779266\n",
    answer="779000",
    reason_if_not="",
)
assert grade_response(test_response)["error_tag"] == "misreported"

test_response.execution_status = "timeout"
assert grade_response(test_response)["error_tag"] == "execution_error"

test_response.question_id = "Q40"
test_response.execution_status = "ok"
assert grade_response(test_response)["error_tag"] == "hallucinated_answer"

print("Stage 4 grading checks passed.")

Stage 4 grading checks passed.


## Stage 5 — Check model outputs

### 5.1 · Apply the rules checker

The checker flags failed executions, unknown column names in direct references
such as `df["column"]`, and disagreements between the written answer and the
printed output. It uses only the model response, execution details, and dataset
column names to decide whether to flag a result.

In [54]:
# Use the column names from the real DataFrame.
known_columns = set(df.columns)


def unknown_columns(code: str, known_columns: set[str]) -> list[str]:
    """Find unknown literal column names in df[...] and df.loc[:, ...]."""
    missing = set()

    for node in ast.walk(ast.parse(code)):
        if not isinstance(node, ast.Subscript):
            continue

        target = node.value
        if isinstance(target, ast.Name) and target.id == "df":
            selection = node.slice
        elif (
            isinstance(target, ast.Attribute)
            and target.attr == "loc"
            and isinstance(target.value, ast.Name)
            and target.value.id == "df"
            and isinstance(node.slice, ast.Tuple)
            and len(node.slice.elts) == 2
        ):
            # In df.loc[rows, columns], the second item selects columns.
            selection = node.slice.elts[1]
        else:
            continue

        # Check literal names such as df["age"] and df[["age", "bike_id"]].
        # A calculated selector such as df[mask] is left to the execution check.
        items = selection.elts if isinstance(selection, ast.List) else [selection]
        for item in items:
            if isinstance(item, ast.Constant) and isinstance(item.value, str):
                if item.value not in known_columns:
                    missing.add(item.value)

    return sorted(missing)


def check_response(result, known_columns: set[str]) -> dict:
    """Flag observable problems without consulting the reference answer."""
    reasons = []
    missing = []

    if result.can_answer:
        if result.execution_status != "ok":
            reasons.append("execution_failed")

        try:
            missing = unknown_columns(result.code, known_columns)
        except SyntaxError:
            # Invalid Python already counts as a failed execution.
            missing = []

        if missing:
            reasons.append("unknown_column")

        if result.execution_status == "ok" and result.answer.strip():
            try:
                # Compare structured values. This rule treats different rounded
                # values as disagreements, even if both could pass grading.
                agrees = parse_value(result.answer) == parse_value(result.stdout)
            except (ValueError, TypeError, SyntaxError, OverflowError):
                agrees = False

            if not agrees:
                reasons.append("answer_stdout_mismatch")

    return {
        "question_id": result.question_id,
        "model": result.model,
        "flagged": bool(reasons),
        "flag_reasons": reasons,
        "unknown_columns": missing,
    }


# Generate flags using only the Stage 3 records and dataset columns.
checker_flags = pd.DataFrame(
    check_response(result, known_columns)
    for result in stage3_results.itertuples(index=False)
)

display(checker_flags)

,question_id,model,flagged,flag_reasons,unknown_columns
0,Q01,mock-v1,False,[],[]
1,Q04,mock-v1,False,[],[]
2,Q40,mock-v1,False,[],[]


### 5.2 · Test the rules and measure checker performance

Test each rule with a controlled example. Then compare the checker’s flags
with the Stage 4 grades. The catch rate is the percentage of incorrect
responses flagged. The false-alarm rate is the percentage of correct
responses flagged.

In the three-response mock run, the checker caught 0 of 1 errors and flagged
0 of 2 correct responses. It missed Q04 because the code ran successfully
and printed a count for the wrong calculation.

In [55]:
from types import SimpleNamespace

# Test each rule with a controlled example.
test_result = SimpleNamespace(
    question_id="test",
    model="checker-test",
    can_answer=True,
    code="print(len(df))",
    execution_status="timeout",
    stdout="",
    answer="",
)
assert check_response(test_result, known_columns)["flag_reasons"] == [
    "execution_failed"
]

test_result.code = 'print(df["rider_age"].mean())'
test_result.execution_status = "execution_error"
check = check_response(test_result, known_columns)
assert "unknown_column" in check["flag_reasons"]
assert check["unknown_columns"] == ["rider_age"]

test_result.code = "print(len(df))"
test_result.execution_status = "ok"
test_result.stdout = "100\n"
test_result.answer = "99"
assert check_response(test_result, known_columns)["flag_reasons"] == [
    "answer_stdout_mismatch"
]

# Bring in the Stage 4 grades only after all flags have been generated.
checker_evaluation = checker_flags.merge(
    graded_results[["question_id", "model", "correct"]],
    on=["question_id", "model"],
    validate="one_to_one",
)
assert len(checker_evaluation) == len(checker_flags) == len(graded_results)

metrics = []
for model, rows in checker_evaluation.groupby("model"):
    incorrect = rows.loc[~rows["correct"]]
    correct = rows.loc[rows["correct"]]
    caught = int(incorrect["flagged"].sum())
    false_alarms = int(correct["flagged"].sum())

    metrics.append({
        "model": model,
        "errors_caught": caught,
        "incorrect_answers": len(incorrect),
        "catch_rate_pct": 100 * caught / len(incorrect) if len(incorrect) else None,
        "false_alarms": false_alarms,
        "correct_answers": len(correct),
        "false_alarm_rate_pct": 100 * false_alarms / len(correct) if len(correct) else None,
    })

checker_metrics = pd.DataFrame(metrics)
display(checker_metrics)
print("Stage 5 checker tests passed.")

,model,errors_caught,incorrect_answers,catch_rate_pct,false_alarms,correct_answers,false_alarm_rate_pct
0,mock-v1,0,1,0.0,0,2,0.0


Stage 5 checker tests passed.
